# Tests de  `cleaner.py`

Este notebook verifica que `cleaner.py` produce el dataset limpio esperado. 

**Prerequisito:** el dataset base debe existir en `data/interim/dataset_base.parquet`. Si no es así, ejecutar primero `01_data_exploration.ipynb`.

## 1. Carga y ejecución del pipeline de limpieza

Se carga el dataset base desde caché y se ejecuta `cargar_dataset_limpio` con `forzar=True` para garantizar que se aplica la versión actual de `cleaner.py` en lugar de leer un artefacto previo. Los logs del proceso muestran en tiempo real las transformaciones aplicadas: columnas eliminadas, filas descartadas y valores fuera de rango convertidos a `NaN`.

In [1]:
from triaje_ia.data.cleaner import cargar_dataset_limpio
import pandas as pd

df = cargar_dataset_limpio(forzar=True)


2026-05-07 19:29:27.919 | WARNING  | triaje_ia.data.loader:cargar_dataset_base:345 - Cargando desde caché: C:\Users\CARLOS\triaje-ia-tfg\data\interim\dataset_base.parquet. Si cambiaste parámetros del pipeline, usa forzar=True.
2026-05-07 19:29:28.403 | SUCCESS  | triaje_ia.data.loader:cargar_dataset_base:350 - Cargado: 425,087 filas | 23 columnas
2026-05-07 19:29:28.404 | INFO     | triaje_ia.data.cleaner:limpiar_dataset:134 - Limpieza: entrada 425,087 filas | 23 columnas
2026-05-07 19:29:28.479 | INFO     | triaje_ia.data.cleaner:limpiar_dataset:144 - Columnas eliminadas: ['hadm_id', 'disposition']
2026-05-07 19:29:28.548 | INFO     | triaje_ia.data.cleaner:limpiar_dataset:150 - Filas sin acuity eliminadas: 6,987 (1.64%)
2026-05-07 19:29:28.578 | INFO     | triaje_ia.data.cleaner:_limpiar_vitales:103 - temperature: 825 valores fuera de [95.0, 107.0] → NaN
2026-05-07 19:29:28.582 | INFO     | triaje_ia.data.cleaner:_limpiar_vitales:103 - heartrate: 29 valores fuera de [20.0, 300.0] → N

## 2. Inspección del dataset limpio

Se inspeccionan las dimensiones finales, los tipos de dato de las columnas clave, el mapa de nulos restantes y la distribución de la variable `race` tras la agrupación. 

In [2]:
# Dimensiones y tipos
print(f"Shape: {df.shape}")
print(f"\nDtipos relevantes:")
print(df[["acuity", "pain", "age", "race"]].dtypes)

# Nulos tras limpieza
nulos = df.isna().sum()
nulos_pct = (nulos / len(df) * 100).round(2)
resumen = pd.DataFrame({
    "n_nulos": nulos,
    "pct": nulos_pct
}).query("n_nulos > 0").sort_values("pct", ascending=False)

print(f"\nNulos tras limpieza:")
print(resumen)

# Race agrupada
print(f"\nRace agrupada:")
print(df["race"].value_counts())

# Vitales — confirmar que no hay imposibles
print(f"\nVitales — rangos tras limpieza:")
vitales = ["temperature", "heartrate", "resprate", "o2sat", "sbp", "dbp"]
print(df[vitales].agg(["min", "max"]).round(1))

Shape: (418100, 21)

Dtipos relevantes:
acuity      int8
pain        Int8
age        Int16
race      object
dtype: object

Nulos tras limpieza:
                n_nulos   pct
pain              32505  7.77
temperature       17423  4.17
o2sat             13930  3.33
resprate          13603  3.25
dbp               12784  3.06
sbp               11686  2.80
heartrate         10316  2.47
ccs_category        972  0.23
age                  75  0.02
chiefcomplaint       17  0.00

Race agrupada:
race
WHITE       244093
BLACK        92168
HISPANIC     35205
OTHER        22136
ASIAN        18321
UNKNOWN       6177
Name: count, dtype: int64

Vitales — rangos tras limpieza:
     temperature  heartrate  resprate  o2sat    sbp    dbp
min         95.0       20.0       4.0   50.0   40.0   10.0
max        107.0      256.0      60.0  100.0  299.0  199.0


## 3. Asserts de validación

Se comprueban mediante `assert` las propiedades invariantes que el dataset limpio debe satisfacer en todo momento. Las comprobaciones cubren: dimensiones exactas del resultado, ausencia de nulos en la variable objetivo `acuity`, eliminación de las columnas descartadas por el pipeline, respeto de los rangos fisiológicos definidos en `RANGOS_VITALES`, categorías válidas en `race`, rango de `pain` entre 0 y 10, y tipos de dato compactos (`int8`, `Int8`, `Int16`) necesarios para la eficiencia en memoria. Cualquier cambio en `cleaner.py` que altere estas propiedades hará fallar este notebook, actuando como test de regresión.

In [3]:
# Asserts de validación

# Dimensiones
assert len(df) == 418_100, f"Filas inesperadas: {len(df)}"  # 425.087 stays iniciales − 6.987 sin acuity = 418.100
assert df.shape[1] == 21, f"Columnas inesperadas: {df.shape[1]}"  # 21 cols: incluye outtime, necesario para feature engineering temporal

# Targets completos
assert df["acuity"].isna().sum() == 0, "acuity tiene nulos"
assert df["acuity"].isin([1,2,3,4,5]).all(), "acuity con valores fuera de [1-5]"

# Columnas eliminadas
for col in ["hadm_id", "disposition"]:
    assert col not in df.columns, f"Columna {col} debería haberse eliminado"
assert "outtime" in df.columns, "outtime debe conservarse (usada en bloque 6 de features)"
assert "medicacion_raw" in df.columns, (
    "medicacion_raw no está en el dataset. "
    "Reinicia el kernel y ejecuta de nuevo con cargar_dataset_limpio(forzar=True)."
)
assert df["medicacion_raw"].isna().sum() == 0, "medicacion_raw tiene NaN"
assert df["medicacion_raw"].dtype == object, "medicacion_raw debe ser string"

# Vitales dentro de rango
from triaje_ia.data.cleaner import RANGOS_VITALES
for col, (vmin, vmax) in RANGOS_VITALES.items():
    assert df[col].dropna().between(vmin, vmax).all(), \
        f"{col} tiene valores fuera de [{vmin}, {vmax}]"

# Race agrupada
assert set(df["race"].dropna().unique()) == \
    {"WHITE", "BLACK", "HISPANIC", "ASIAN", "OTHER", "UNKNOWN"}, \
    "race tiene categorías inesperadas"

# Pain en rango 0-10
assert df["pain"].dropna().between(0, 10).all(), "pain tiene valores fuera de [0,10]"

# Tipos
assert str(df["acuity"].dtype) == "int8"
assert str(df["pain"].dtype)   == "Int8"
assert str(df["age"].dtype)    == "Int16"

print("Todos los asserts pasaron")

Todos los asserts pasaron
